# Function Calling & Tool Use

LLM不是什么活都能干。它的全部能耐就是生成文本，所以不能获取天气、查询数据库、发送Email、跑代码或者阅读文件。所有你见过的AI Agent都是LLM生成了应该调用哪个函数的JSON，然后你的代码去执行。模型是大脑，工具是手，Function calling 就是连接它们的神经系统。

## 问题描述

当你想大模型询问天气是，正确的答案需要调用获取天气的API。但是模型不能做这个事，你的代码可以。缺的是：一个结果化的协议，让你的模型返回“我需要用这些参数调用API”，然后你的代码执行，之后把获取的结果喂回去。

这就是Function Calling。模型输出标准化的JSON，描述应该以哪些参数调用哪个模型。你的应用执行对应函数，获取到的结果喂回给会话，然后模型基于此产生最终答案。

## 基本概念

### Funtion Calling 循环
```mermaid
sequenceDiagram
participant A as User
participant B as App
participant C as Model
participant D as Tool

A->>B: 发起问询
B->>C: 信息+工具定义
C->>B: tool_call ...
B->>D: execute
D->>B: result
B->>C: tool result + 会话
C->>B: 回复
B->>A: 最后答案
```

### 工具定义

```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather for a city. Returns temperature in Celsius and conditions.",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name, e.g. 'Tokyo' or 'San Francisco'"
        },
        "units": {
          "type": "string",
          "enum": ["celsius", "fahrenheit"],
          "description": "Temperature units"
        }
      },
      "required": ["city"]
    }
  }
}
```

最关键的字段是`decription`。模型读取它们来决定什么时候用哪个工具。

### 供应商对比

每个主要的大模型供应商都支持function calling，但是API 不同。
|供应商|API参数|工具调用格式|并行|强制调用|
|---|---|---|---|---|
|OpenAI|tools|tool_calls[].function|Yes|tool_choice="required"|
|Anthropic|tools|content[].type="tool_use"|Yes|tool_choice={"type":"any"}|
|Google|function_declarations|functionCall|Yes|function_call_config|
|Ohter|Mixed|Mixed|Model-dependent|Prompt-based or tool_choice|

### 工具选择 Auto、Required、Specific

你可以控制模型什么时候使用工具

- Auto(default)。 模型决定是否调用工具，或者直接回复。
- Required。 模型必须至少调用一个工具，当你知道用户的意图需要工具的时候使用。防止模型不看实际数据就猜答案。
- Specific。 强制模型输出特别的工具调用。`tool_choice={"type": "function", "function": {"name": "get_weather"}}`保证天气工具被调用，不管输入是什么。

### 并行Function Calling

模型可以一轮输出多个Function Calling
```
{
    {"name", "...", "arguments": {"key": "value"}},
    {"name", "...", "arguments": {"key": "value"}},
}
```

理想情况下你的模型并发地执行所有，返回答案，然后综合成一条回复。

### 结构化输出和Function Calling

Function Calling的JSON模式机制与结构化输出一致，但是目的不一样。

结构化输出是为了强制模型按照特定的形状输出结果。模型输出就是最后的产物。

Function calling是模型表达行动的方式。输出结果是一个中间步。

在你需要做数据抽取的时候使用结构化输出，在你需要与外部系统交互的时候使用Function Calling。

### 安全性

Function Calling 是你给大模型最危险的能力，模型决定执行什么。如果你的工具中包含Shell命令，模型就能写它。

- 永远不要把模型生成的SQL直接用于数据库。
- 使用白名单。
- 校验输入参数。
- 清理工具输出的结果。比如API keys等。
- 控制工具调用次数

### 错误处理

工具失败了，API超时，数据库下线，文件不存在。模型需要知道什么时候工具调用失败以及为什么。将错误按照结果化的工具结果返回，而不是异常。
```json
{
    "error": true,
    "message": "...",
    "code": "INVALID_ARGUMENT ..."
}
```

模型独到它，然后调整参数，之后重试。对于结构化的失败信息，模型很擅长自我纠正。但是他们对于空的回复以及“有什么出错了”这种错误很难修复。

### 模型上下文协议（MCP）

MCP 是Anthropic为工具互通性做的开放标准。不是每个应用都自己去定义工具，MCP提供一个通用协议：工具由MCP 服务器提供，由MCP 客户端（比如ClaudeCode，Cursor或者你的应用）消费。

一个MCP 服务器可以暴露工具给任何兼容的客户端。它标准化了传输层，所以做到了工具可移植性。



# 开始编码

## 工具定义示例代码

In [4]:
import json
import math
import time

TOOL_REGISTRY = {}

def register_tool(
    name: str,
    description: str,
    parameters: dict,
    function: str
) -> None:
    TOOL_REGISTRY[name] = {
        "definition": {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": parameters,
            }
        },
        "function": function,
    }

def weather_tool(city: str, units: str = "celsius") -> str:
    """Get current weather for a city. Returns temperature in Celsius and conditions."""
    return f"The weather in {city} is 20°C and cloudy."

register_tool(
    "weather",
    "Get current weather for a city. Returns temperature in Celsius and conditions.",
    {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g. 'Tokyo' or 'San Francisco'"
            },
            "units": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature units"
            }
        },
        "required": ["city"]
    },
    weather_tool
)

def code_tool(language: str, code: str) -> str:
    """Execute code in a specific language."""
    if language != "python":
        return {"error": True, "message": f"Unsupported language {language}, Only Python is supported."}

    forbidden = [
        "import os",
        "import sys"
    ]

    for pattern in forbidden:
        if pattern in code:
            return {"error": True, "message": f"Forbidden import {pattern}", "code": "SECURITY_VIOLATION"}

    try:
        local_vars = {}
        exec(
            code,
            {},
            local_vars
        )

        result = local_vars.get("result", None)
        return {
            "success": True,
            "result": result,
            "variables": {
                k:str(v) for k, v in local_vars.items() if not k.startswith("_")
            }
        }

    except Exception as e:
        return {"error": True, "message": f"{type(e).__name__}: {e}"}
        

## 辅助示例代码

In [ ]:
def execute_tool_call(tool_call):
    name = tool_call["name"]
    args = tool_call["arguments"]

    if name not in TOOL_REGISTRY:
        return {
            "tool": name,
            "result": {
                "error": True,
                "message": f"Tool {name} not found",
                "code": "TOOL_NOT_FOUND"
            },
            "execution_time_ms": 0
        }

    tool_def = TOOL_REGISTRY[name]
    function = tool_def["function"]
    start = time.time()

    try:
        result = function(**args)
    except TypeError as e:    
        result = {
            "error": True,
            "message": f"Invalid arguments: {e}",
            "code": "INVALID_ARGUMENTS"
        }

    elapsed_ms = round((time.time() - start) * 1000, 2)

    return {
        "tool": name,
        "result": result,
        "execution_time_ms": elapsed_ms
    }        


# API

In [ ]:
from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

import sys
from pathlib import Path
from rich import print as rprint

sys.path.append(str(Path("../../00_Common").resolve()))

from user_tools import SectionPrinter, load_project_env

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"

# create_agent 返回 CompiledStateGraph，没有 bind/bind_tools；
# 工具必须在创建时传入可调用对象（或 BaseTool）。
tools = [
    StructuredTool.from_function(
        func=entry["function"],
        name=entry["definition"]["function"]["name"],
        description=entry["definition"]["function"]["description"],
    )
    for entry in TOOL_REGISTRY.values()
]

agent = create_agent(
    init_chat_model(
        MODEL,
        extra_body={"thinking": {"type": "disabled"}},
    ),
    tools=tools,
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]}
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content="What's the weather in Tokyo?",
            additional_kwargs={},
            response_metadata={},
            id='dd252407-9ce8-461f-bcf1-eda4afebca57'
        ),
        AIMessage(
            content="I'll check the current weather in Tokyo for you.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 52,
                    'prompt_tokens': 302,
                    'total_tokens': 354,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 46
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '3d1a4d12-0d83-4a94-aff2-67d433861abe',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a01562-fd31-7d23-bdb0-f1929640070f-0',
            tool_calls=[
                {
                    'name': 'weather',
                    'args': {'city': 'Tokyo'},
                    'id': 'call_00_dYPmd8Bdg2009JZFEgQg1481',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 302,
                'output_tokens': 52,
                'total_tokens': 354,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='The weather in Tokyo is 20°C and cloudy.',
            name='weather',
            id='ec71165d-a790-4ac8-81ac-e524dec7c50d',
            tool_call_id='call_00_dYPmd8Bdg2009JZFEgQg1481'
        ),
        AIMessage(
            content='The weather in Tokyo is currently **20°C and cloudy**. ☁️',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 16,
                    'prompt_tokens': 376,
                    'total_tokens': 392,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 256
                    },
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 120
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'b94e1a33-af62-43f5-88e4-c85832aba472',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a01563-05f7-7ce3-a2c3-7425ba736fba-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 376,
                'output_tokens': 16,
                'total_tokens': 392,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        )
    ]
}

AttributeError: 'dict' object has no attribute 'choices'